# Prepare Re-Verification Batches

Reads all JSON files from `prompts_batches_20/batch_*/` and writes a new folder `prompts_batches_20_reverif/`  
with the same batch organisation (`batch_01/` … `batch_20/`), keeping only:
- `id`
- `specie`
- `specie_en`
- `corrected_text`  ← extracted from `verification.corrected_text`

In [1]:
import json
from pathlib import Path

In [2]:
batches_dir  = Path("../prompts_batches_20")
output_dir   = Path("../prompts_batches_20_reverif")

batch_folders = sorted(batches_dir.glob("batch_*"))
print(f"Found {len(batch_folders)} batch folders in '{batches_dir}'")

Found 20 batch folders in '../prompts_batches_20'


In [3]:
copied  = 0
skipped = 0
missing_fields = []

for batch_folder in batch_folders:
    out_batch = output_dir / batch_folder.name
    out_batch.mkdir(parents=True, exist_ok=True)

    for src in sorted(batch_folder.glob("*.json")):
        data = json.loads(src.read_text(encoding="utf-8"))

        corrected_text = data.get("verification", {}).get("corrected_text")

        if corrected_text is None:
            missing_fields.append(str(src))
            skipped += 1
            continue

        record = {
            "id":             data.get("id"),
            "specie":         data.get("specie"),
            "specie_en":      data.get("specie_en"),
            "corrected_text": corrected_text,
        }

        dst = out_batch / src.name
        dst.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
        copied += 1

print(f"Done: {copied} files written, {skipped} skipped (missing corrected_text)")
print(f"Output folder: {output_dir.resolve()}")

if missing_fields:
    print("\nFiles with missing corrected_text:")
    for f in missing_fields:
        print(f"  {f}")

Done: 7277 files written, 4 skipped (missing corrected_text)
Output folder: /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/prompts_batches_20_reverif

Files with missing corrected_text:
  ../prompts_batches_20/batch_04/1414_Chloris_chloris_std.json
  ../prompts_batches_20/batch_13/4791_Osmerus_eperlanus_std.json
  ../prompts_batches_20/batch_17/6089_Saurida_tumbil_abj.json
  ../prompts_batches_20/batch_18/6354_Setophaga_dominica_std.json


In [4]:
# Fallback: for files skipped due to missing corrected_text, use generated_text instead
fallback_copied  = 0
fallback_skipped = 0
still_missing    = []

for src_path in missing_fields:
    src  = Path(src_path)
    data = json.loads(src.read_text(encoding="utf-8"))

    generated_text = data.get("generated_text")

    if generated_text is None:
        still_missing.append(src_path)
        fallback_skipped += 1
        continue

    record = {
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),
        "corrected_text": generated_text,   # filled with generated_text as fallback
    }

    # mirror the same batch subfolder in the output
    relative = src.relative_to(batches_dir)          # e.g. batch_13/4791_Osmerus_eperlanus_std.json
    dst = output_dir / relative
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    fallback_copied += 1

print(f"Fallback: {fallback_copied} files written using generated_text, {fallback_skipped} still skipped")
if still_missing:
    print("\\nFiles with neither corrected_text nor generated_text:")
    for f in still_missing:
        print(f"  {f}")

Fallback: 4 files written using generated_text, 0 still skipped


In [5]:
# Sanity check: compare file counts per batch
print(f"{'Batch':<12} {'Source':>8} {'Output':>8} {'Match':>6}")
print("-" * 38)
for batch_folder in sorted(batches_dir.glob("batch_*")):
    src_count = len(list(batch_folder.glob("*.json")))
    out_count = len(list((output_dir / batch_folder.name).glob("*.json")))
    match = "OK" if src_count == out_count else "MISMATCH"
    print(f"{batch_folder.name:<12} {src_count:>8} {out_count:>8} {match:>6}")

Batch          Source   Output  Match
--------------------------------------
batch_01          370      370     OK
batch_02          370      370     OK
batch_03          370      370     OK
batch_04          370      370     OK
batch_05          370      370     OK
batch_06          370      370     OK
batch_07          370      370     OK
batch_08          370      370     OK
batch_09          370      370     OK
batch_10          370      370     OK
batch_11          370      370     OK
batch_12          370      370     OK
batch_13          370      370     OK
batch_14          370      370     OK
batch_15          370      370     OK
batch_16          370      370     OK
batch_17          370      370     OK
batch_18          370      370     OK
batch_19          370      370     OK
batch_20          251      251     OK
